# MURI: Modelling actuator self-assembly

## Import surface mesh and compute fiber orientation

First, import statements

In [1]:
import numpy as np
import fabsim_py
import pyvista as pv
import tetgen

Constants

In [2]:
dt = 1 / 24
k0 = 1e-4
k1 = 5e-2
kd = 0.1 # rate of dissociation
e0 = 1.2e-1
e1 = 1.7e-1
frac_f = 0.7
frac_s = 0.25
n = 16

Fix DOFs

In [3]:
fixed_dofs = []

def fix_dofs_in_circle(radius, center, V):
  for i in range(V.shape[0]):
    if np.linalg.norm(center - V[i, :2]) < radius + 1e-6:
      fixed_dofs.append(3 * i)
      fixed_dofs.append(3 * i + 1)

In [4]:
mesh = pv.read("top_surface.obj")
P = np.array(mesh.points, dtype=np.float64)
F = np.array(mesh.faces, dtype=np.int32)
F = F.reshape((F.shape[0] // 4, 4))[:, 1:]

fixed_dofs = []
fix_dofs_in_circle(0.75, np.array([-4.55/2, 0]), P)
fix_dofs_in_circle(0.75, np.array([4.55/2, 0]), P)

Steady-state solution for fiber distribution

In [5]:
stretch_factor = 1.5
polymer_frac = np.zeros((F.shape[0], n))
V = P.copy()

for i in range(10):
  fabsim_py.simulate_membrane(V, P, F, polymer_frac, fixed_dofs, stretch_factor, 0.25, 1.0, e0, e1)
  stress = fabsim_py.fiber_stress(V, P / stretch_factor, F, n, e0, e1)
  polymer_frac = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)


## Generate tet mesh and simulate compaction

In [6]:
pv.set_plot_theme('document')

mesh = pv.read("tissue.obj")
tet = tetgen.TetGen(mesh)
V_3D, F_3D = tet.tetrahedralize(order=1, mindihedral=20, minratio=1.5)
grid = tet.grid
grid.plot(show_edges=True)


Widget(value='<iframe src="http://localhost:58684/index.html?ui=P_0x11c6c8c80_0&reconnect=auto" class="pyvista…

In [7]:
fixed_dofs = []
fix_dofs_in_circle(0.75, np.array([-4.55/2, 0]), V_3D)
fix_dofs_in_circle(0.75, np.array([4.55/2, 0]), V_3D)

NV_3D = V_3D.copy()
Phi_3D = np.zeros((F_3D.shape[0], n))
fabsim_py.simulate3D(NV_3D, V_3D, F_3D, Phi_3D, fixed_dofs, 1.3, 0.3, 1, e0, e1)
fabsim_py.simulate3D(NV_3D, V_3D, F_3D, Phi_3D, fixed_dofs, 1.4, 0.3, 1, e0, e1)
fabsim_py.simulate3D(NV_3D, V_3D, F_3D, Phi_3D, fixed_dofs, 1.5, 0.3, 1, e0, e1)

cells = np.hstack((np.full((F_3D.shape[0], 1), 4), F_3D[:])).flatten()
celltypes = np.full(F_3D.shape[0], pv.CellType.TETRA)
points = NV_3D
grid = pv.UnstructuredGrid(cells, celltypes, points)
grid.plot(show_edges=True)


Initial energy: 24210.8
Decrement in iteration 0: 27830.1	Factorization = Exact
Decrement in iteration 1: 2422.42	Factorization = Project
Decrement in iteration 2: 25.8658	Factorization = Project
Decrement in iteration 3: 0.00791387	Factorization = Exact
Decrement in iteration 4: 5.83187e-08	Factorization = Exact
Final energy: 757.362
Initial energy: 3822.82
Decrement in iteration 0: 2300	Factorization = Exact
Decrement in iteration 1: 15.776	Factorization = Project
Decrement in iteration 2: 0.00159295	Factorization = Exact
Decrement in iteration 3: 1.40448e-08	Factorization = Exact
Final energy: 1639.99
Initial energy: 5398.07
Decrement in iteration 0: 2433.86	Factorization = Exact
Decrement in iteration 1: 14.7998	Factorization = Project
Decrement in iteration 2: 1.46572	Factorization = Exact
Decrement in iteration 3: 0.124869	Factorization = Exact
Decrement in iteration 4: 0.00168357	Factorization = Exact
Decrement in iteration 5: 3.83431e-07	Factorization = Exact
Final energy: 3080

Widget(value='<iframe src="http://localhost:58684/index.html?ui=P_0x16ae3fda0_1&reconnect=auto" class="pyvista…

Transfer polymer fractions to 3D mesh

In [8]:
Phi_3D = fabsim_py.transfer_data_to_3D_mesh(P, F, polymer_frac, V_3D, F_3D)
import sys
np.set_printoptions(threshold=sys.maxsize)
print(Phi_3D.shape, F_3D.shape)
# print(Phi_3D)

(5337, 16) (5337, 4)


In [9]:
fixed_dofs = []
fix_dofs_in_circle(0.75, np.array([-4.55/2, 0]), V_3D)
fix_dofs_in_circle(0.75, np.array([4.55/2, 0]), V_3D)

NV_3D = V_3D.copy()
fabsim_py.simulate3D(NV_3D, V_3D, F_3D, Phi_3D, fixed_dofs, 1.3, 0.3, 1, e0, e1)
fabsim_py.simulate3D(NV_3D, V_3D, F_3D, Phi_3D, fixed_dofs, 1.5, 0.3, 1, e0, e1)

cells = np.hstack((np.full((F_3D.shape[0], 1), 4), F_3D[:])).flatten()
celltypes = np.full(F_3D.shape[0], pv.CellType.TETRA)
points = NV_3D
grid = pv.UnstructuredGrid(cells, celltypes, points)
grid.plot(show_edges=True)

Initial energy: 46390.7
Decrement in iteration 0: 35835.2	Factorization = Exact
Decrement in iteration 1: 5452.26	Factorization = Exact
Decrement in iteration 2: 591.883	Factorization = Exact
Decrement in iteration 3: 20.4785	Factorization = Exact
Decrement in iteration 4: 0.319731	Factorization = Exact
Decrement in iteration 5: 0.000793751	Factorization = Exact
Decrement in iteration 6: 9.35993e-09	Factorization = Exact
Final energy: 3336.7
Initial energy: 41325.8
Decrement in iteration 0: 14109.2	Factorization = Exact
Decrement in iteration 1: 837.388	Factorization = Exact
Decrement in iteration 2: 18.2715	Factorization = Exact
Decrement in iteration 3: 0.100446	Factorization = Exact
Decrement in iteration 4: 2.35826e-05	Factorization = Exact
Decrement in iteration 5: 2.45553e-12	Factorization = Exact
Final energy: 26050.7


Widget(value='<iframe src="http://localhost:58684/index.html?ui=P_0x16a2a4320_2&reconnect=auto" class="pyvista…

In [ ]:
# import polyscope as ps

# ps.init()
# ps_vol = ps.register_volume_mesh("sim volume mesh", V_3D, tets=F_3D)
# ps_vol.add_scalar_quantity("data", Phi_3D[:,0], defined_on="cells")
# ps.register_surface_mesh("2D mesh", P, F)
# ps.show()


: 